# K-Means Clustering from Scratch in 3D

### Euclidean Distance, Centroid Updates, Convergence, and the Elbow Method

This notebook implements the core mechanics of **K-Means clustering from scratch** rather than relying on a library estimator. The workflow builds reusable functions for distance calculation, centroid initialization, cluster assignment, centroid recomputation, iterative convergence, and selection of \(k\) through an elbow-style analysis.

### Workflow

1. Load and visualize a three-dimensional synthetic clustering dataset.
2. Implement Euclidean distance.
3. Randomly initialize \(k\) cluster centers.
4. Assign each point to its nearest center and recompute centroids.
5. Repeat until the total-distance objective stabilizes.
6. Compare candidate values of \(k\).

**Data availability:** The original `cluster_blobs_3d.csv` file is not included in this portfolio archive. The notebook therefore preserves the outputs from its original execution and should not be treated as fully reproducible without the source CSV.

In [12]:

import random
import numpy as np

# Set a fixed seed for reproducibility
seed = 1125
random.seed(seed)
np.random.seed(seed)

## 1. Dataset and Initial Visualization

The source data contains three numeric coordinates (`x`, `y`, and `z`). The original run first visualized the point cloud without cluster labels to establish the structure of the data.

In [13]:
import pandas as pd
import plotly.graph_objects as go
import numpy.testing as npt

# Load the 3D CSV file
df = pd.read_csv('cluster_blobs_3d.csv')

# Create 3D scatter without showing labels
fig = go.Figure(data=[go.Scatter3d(
    x=df['x'],
    y=df['y'],
    z=df['z'],
    mode='markers',
    marker=dict(
        size=4,
        color='black',
        opacity=0.6
    ),
    hoverinfo='skip'  # <== disables hover text (no labels)
)])

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
    title="3D Scatter Plot (No Labels)",
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


## 2. Euclidean Distance

For two \(d\)-dimensional points \(x\) and \(y\), Euclidean distance is

\[
d(x,y)=\sqrt{\sum_{i=1}^{d}(x_i-y_i)^2}.
\]

The helper below is dimension-agnostic and is validated against a known three-dimensional example.

In [14]:
import math

def euclidean_distance(point1, point2):
    # point1 is a tuple of coordinates
    # point is a tumple of coordinates
    # should be able to handle
    # Compute the sum of the squared differences for each coordinate
    squared_sum = sum((a - b) ** 2 for a, b in zip(point1, point2))
    # Return the square root of the summed squared differences
    return math.sqrt(squared_sum)

In [15]:
## ASSERT. DO NOT DELETE

p1 = (0, 0, 0)
p2 = (3, 4, 5)
expected = np.sqrt(50)
npt.assert_almost_equal(euclidean_distance(p1, p2), expected, decimal=6)

## 3. Random Centroid Initialization

The initial centroids are sampled directly from the observations. A fixed random seed is used in the notebook setup so the archived run is deterministic.

In [16]:
def choose_random(n, dataset):
    # Choose random points, return it as a dictionary
    random_points = random.sample(dataset, n)
    # Construct dictionary with keys starting from 1
    return {i+1: point for i, point in enumerate(random_points)}

In [17]:
dataset = [
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 9),
    (10, 11, 12),
    (13, 14, 15)
]

n = 3
result = choose_random(n, dataset)

# ASSERT DO NOT DELETE
assert isinstance(result, dict) and len(result) == n
assert all(isinstance(k, int) and 1 <= k <= n and isinstance(v, tuple) and len(v) == 3
           for k, v in result.items())

## 4. Cluster Assignment and Centroid Update

Each observation is assigned to its nearest centroid. New centroids are then computed as the coordinate-wise mean of all observations assigned to each cluster. If a cluster receives no observations, its previous center is retained.

In [18]:
def label_cluster(dataset, cluster_center, iteration=1):
# INSERT CODE HERE
    
    labels = []
    total_distance = 0.0
    # Dictionary to hold the points assigned to each cluster
    assignments = {cid: [] for cid in cluster_center.keys()}
    
    # Label each point with the closest cluster center
    for point in dataset:
        best_cluster = None
        best_distance = float('inf')
        for cid, center in cluster_center.items():
            # Compute Euclidean distance between point and cluster center using the euclidean_distance function
            d = euclidean_distance(point, center)
            if d < best_distance:
                best_distance = d
                best_cluster = cid
        labels.append(best_cluster)
        total_distance += best_distance
        assignments[best_cluster].append(point)
    
    # Compute new cluster centers based on the average of points assigned to each cluster
    new_cluster_center = {}
    for cid, points in assignments.items():
        if points:  # Compute the mean if there are assigned points
            dims = len(points[0])
            avg_point = tuple(sum(point[i] for point in points) / len(points) for i in range(dims))
            new_cluster_center[cid] = avg_point
        else:
            # If no points were assigned to this cluster, retain the previous center
            new_cluster_center[cid] = cluster_center[cid]
    
    return labels, total_distance, new_cluster_center

In [19]:
# ASSERT (DO NOT DELETE)
dataset = [(1, 2, 3), (4, 5, 6), (1, 1, 1), (10, 10, 10)]
cluster_center = {1: (1, 2, 3), 2: (10, 10, 10)}

labels, total_dist, new_centers = label_cluster(dataset, cluster_center, iteration=1)

assert isinstance(labels, list) and len(labels) == len(dataset)
assert isinstance(total_dist, (int, float))

assert isinstance(new_centers, dict) and set(new_centers.keys()) == set(cluster_center.keys())
for v in new_centers.values():
    assert isinstance(v, tuple) and len(v) == 3

## 5. Iterative K-Means Procedure

The full routine repeats assignment and centroid recomputation until either:

- the relative change in total distance falls below the specified tolerance; or
- the maximum number of iterations is reached.

In this implementation, the objective tracked for convergence is the **sum of Euclidean distances to assigned centroids**. This differs from the squared-distance inertia commonly reported by scikit-learn's `KMeans`.

In [20]:
import plotly.express as px

def kMeans_Clustering(dataset, k=3, tolerance=0.01, max_iter=100):
    cluster_center = choose_random(k, dataset)
    prev_total_distance = float('inf')
    labels = []

    for i in range(1, max_iter + 1):
        labels, total_distance, cluster_center = label_cluster(dataset, cluster_center, iteration=i)

        # Stop if percent change is less than tolerance
        if i > 1:
            percent_change = abs(prev_total_distance - total_distance) / prev_total_distance
            if percent_change <= tolerance:
                print(f"Converged at iteration {i} with percent change {percent_change:.4f}")
                break

        prev_total_distance = total_distance

    return labels, cluster_center, total_distance

## 6. Four-Cluster Solution

The stored run applies the implementation with \(k=4\). It converged after **4 iterations**, with a relative total-distance change of approximately **0.0064**, and the resulting assignments are shown in the preserved 3D visualization.

In [21]:
## ASSERT (YOU SHOULD BE ABLE TO REPLICATE THIS)

df = pd.read_csv('cluster_blobs_3d.csv')
dataset = [tuple(x) for x in df[['x', 'y', 'z']].values]
labels, centers, total_distance = kMeans_Clustering(dataset, k=4, tolerance=0.01)

# Assign predicted labels back to DataFrame
df["predicted_label"] = labels

# Plot result
fig = px.scatter_3d(df, x="x", y="y", z="z", color=df["predicted_label"].astype(str),
                    title="K-Means Clustering on cluster_blobs_3d.csv", opacity=0.7)
fig.show()

Converged at iteration 4 with percent change 0.0064


## 7. Choosing the Number of Clusters

The implementation is evaluated from \(k=2\) through \(k=10\). The archived curve shows its strongest reduction in total distance up to approximately **\(k=4\)**, after which improvements become smaller overall.

Because each value of \(k\) is evaluated using a single random initialization, the objective is not guaranteed to decrease monotonically as \(k\) increases. In a production implementation, multiple initializations per \(k\) would provide a more stable comparison.

In [22]:
### INSERT CODE HERE

# Run kMeans_Clustering for k values from 2 to 10 and store total_distance for each k
k_values = list(range(2, 11))
total_distances = []

for k in k_values:
    print(f"Running kMeans Clustering for k = {k}...")
    labels, centers, total_distance = kMeans_Clustering(dataset, k=k, tolerance=0.01)
    total_distances.append(total_distance)

# Create a DataFrame for plotting
plot_df = pd.DataFrame({'Number of Clusters (k)': k_values, 'Total Distance': total_distances})

# Plot the total_distance as a function of k using a line plot
fig = px.line(plot_df, x='Number of Clusters (k)', y='Total Distance', markers=True,
              title="Total Distance vs. Number of Clusters (k)",
              labels={'Total Distance': 'Total Distance'})
fig.show()


Running kMeans Clustering for k = 2...
Converged at iteration 3 with percent change 0.0011
Running kMeans Clustering for k = 3...
Converged at iteration 3 with percent change 0.0020
Running kMeans Clustering for k = 4...
Converged at iteration 4 with percent change 0.0036
Running kMeans Clustering for k = 5...
Converged at iteration 4 with percent change 0.0098
Running kMeans Clustering for k = 6...
Converged at iteration 6 with percent change 0.0014
Running kMeans Clustering for k = 7...
Converged at iteration 4 with percent change 0.0098
Running kMeans Clustering for k = 8...
Converged at iteration 4 with percent change 0.0088
Running kMeans Clustering for k = 9...
Converged at iteration 4 with percent change 0.0081
Running kMeans Clustering for k = 10...
Converged at iteration 6 with percent change 0.0044


## 8. Key Takeaways

- The notebook reconstructs K-Means using basic Python and NumPy-compatible operations.
- The stored elbow-style analysis supports **\(k=4\)** as a reasonable cluster count for this dataset.
- Convergence is based on relative change in total Euclidean distance.
- A stronger production implementation would use multiple random restarts, report squared-distance inertia, and compare cluster-quality metrics such as silhouette score.

This project is best viewed as an implementation exercise demonstrating the mechanics behind centroid-based clustering rather than as a production clustering pipeline.